soubor obsahuje ukázku jak načíst model, neobsahuje ukázku jak načíst data, to je v notebooku detection_model.ipynb

In [ ]:
import torch
import torch.nn as nn

class ConcatBlockConv5(nn.Module):
    def __init__(self, in_ch, out_ch, k=32, act=nn.SiLU):
        super().__init__()
        
        def make_block(kernel_size):
            return nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding='same'),
                nn.BatchNorm1d(out_ch),
                act()
            )

        self.c1 = make_block(k)          # k=32
        self.c2 = make_block(k * 2)      # k=64
        self.c3 = make_block(k // 2)     # k=16
        self.c4 = make_block(k // 4)     # k=8
        self.c5 = make_block(k * 4)      # k=128
        
        c6_in_channels = (out_ch * 5) + in_ch
        
        self.c6 = nn.Sequential(
            nn.Conv1d(c6_in_channels, out_ch, kernel_size=1),
            nn.BatchNorm1d(out_ch),
            act()
        )

    def forward(self, x):
        x1 = self.c1(x)
        x2 = self.c2(x)
        x3 = self.c3(x)
        x4 = self.c4(x)
        x5 = self.c5(x)

        out = torch.cat([x1, x2, x3, x4, x5, x], dim=1)
        
        return self.c6(out)

In [ ]:
class GWNet(nn.Module):
    def __init__(self, channels=3, classes=1):
        super().__init__()

        self.b1 = ConcatBlockConv5(channels, 32, k=32)
        self.p1 = nn.MaxPool1d(2)

        self.b2 = ConcatBlockConv5(32, 64, k=32)
        self.p2 = nn.MaxPool1d(2)

        self.b3 = ConcatBlockConv5(64, 128, k=32)
        self.p3 = nn.MaxPool1d(2)

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(128, classes)

    def forward(self, x):
        x = self.p1(self.b1(x))
        x = self.p2(self.b2(x))
        x = self.p3(self.b3(x))
        x = self.gap(x).squeeze(-1)
        return self.fc(x)

In [ ]:
model = GWNet()
checkpoint_path = "residual_model_least_process_final.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=torch.device('cpu')))
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Model loaded successfully via state_dict!")

In [ ]:
def read_file(fname):
    data = np.load(fname)
    return data

from scipy import signal
from scipy.signal.windows import tukey
window = tukey(4096, 0.1)
sos = signal.butter(8, [20, 500], btype="bandpass", output="sos", fs=2048)
def bandpass(x):
    x *= window
    for i in range(3):
        x[i] = signal.sosfilt(sos, x[i])
    return x

scaling = [1.5e-20, 1.5e-20, 0.5e-20]
def preprocess(data):
    p1, p2, p3 = bandpass(data)
    p1 = p1 / scaling[0]
    p2 = p2 / scaling[1]
    p3 = p3 / scaling[2]
    return p1, p2, p3

def load_and_preprocess(file_path):
    data = read_file(file_path)
    p1, p2, p3 = preprocess(data, 0)
    stacked = np.stack([p1, p2, p3], axis=0)
    return stacked

def get_model_probability(processed_data):
    input_tensor = torch.tensor(processed_data).float()
    input_tensor = input_tensor.unsqueeze(0)
    device = next(model.parameters()).device
    input_tensor = input_tensor.to(device)
    with torch.no_grad():
        outputs = model(input_tensor)
        prob = torch.sigmoid(outputs).item()
    return prob